# 9C · Forecasting a Seasonal Series — The Full Harness, Deployed
### Financial Analytics — Module 9 · Lab 1

The lab's finale: forecast MoneyMart's monthly sales properly, using everything Module 6 built:

1. The right **floor**: for seasonal data, naive isn't naive enough — meet **seasonal naive**
2. **Holt-Winters** exponential smoothing: level + trend + season, learned together
3. A **gentle ARIMA** — fit-and-forecast, zero theory-terror
4. **Walk-forward** judgment and **intervals that widen with horizon** (the forecast-slider chart)

> 🛡️ **Bias check:** the sales series is synthetic and documented — no survivorship, no restatements, and (unless you did 9A's Exercise 3) no breaks. Every fit below trains strictly on the past. Look-ahead: guarded by construction; verify the split dates yourself anyway.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Rebuild 9A's series - same seed, byte-identical (auditability in action)
rng = np.random.default_rng(9)
months = pd.date_range("2020-01-31", periods=72, freq="ME")
true_trend = 520 + 4.2*np.arange(72)
seasonal_index = {1:0.94, 2:0.92, 3:0.98, 4:1.00, 5:0.97, 6:0.95,
                  7:0.98, 8:1.02, 9:1.05, 10:1.22, 11:1.14, 12:0.83}
true_seasonal = np.array([seasonal_index[m.month] for m in months])
sales = pd.Series(true_trend*true_seasonal*rng.normal(1.0, 0.03, 72), index=months, name="sales_cr")

split = "2024-12-31"
train, test = sales[:split], sales[split:][0:]
train, test = sales.iloc[:60], sales.iloc[60:]      # 5 years train, 1 year test
print(f"Train: {len(train)} months to {train.index[-1].date()} | Test: {len(test)} months")

---
## 1. The floor, upgraded: seasonal naive

Plain naive ("next month = this month") is a poor floor for seasonal data — it's *systematically* wrong every October and December. The honest floor is **seasonal naive: next month = the same month last year.** It costs nothing and already knows the festive rhythm. Any real model must beat *this*.

In [ ]:
def eval_forecast(pred, actual, name):
    e = actual - pred
    return pd.Series({"MAE": e.abs().mean(),
                      "RMSE": np.sqrt((e**2).mean()),
                      "MAPE%": (e.abs()/actual).mean()*100}, name=name)

f_naive  = pd.Series(train.iloc[-1], index=test.index)                       # flat line at last value
f_snaive = pd.Series(train.iloc[-12:].values, index=test.index)              # same month, last year

rows = [eval_forecast(f_naive, test, "naive"), eval_forecast(f_snaive, test, "seasonal naive")]
print(pd.DataFrame(rows).round(1))
print()
print("Seasonal naive crushes plain naive - the calendar alone is worth that much.")
print("THE FLOOR for this series is now seasonal naive. Everything below must beat IT.")

---
## 2. Holt-Winters: smoothing that knows about trend and season

Module 6's SES tracked a level with one dial. **Holt-Winters** runs three smoothers at once — level, trend, and twelve seasonal factors — each updating as data arrives. Same idea, three dials. We use statsmodels' implementation (you built SES by hand in 6B; you've earned the library):

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

hw = ExponentialSmoothing(train, trend="add", seasonal="mul", seasonal_periods=12).fit()
f_hw = hw.forecast(len(test))

oct_fc = float(f_hw[f_hw.index.month == 10].iloc[0])
dec_fc = float(f_hw[f_hw.index.month == 12].iloc[0])
print(f"Forecast Oct-25: Rs {oct_fc:,.0f} cr vs Dec-25: Rs {dec_fc:,.0f} cr")
print("The festive shape survives into the forecast - the model learned 9A's season on its own.")
print()
print(eval_forecast(f_hw, test, "Holt-Winters").round(1).to_string())

---
## 3. A gentle ARIMA — fit and forecast, no terror

**ARIMA(p, d, q)** in one honest paragraph: *d* differences the series until it's stationary (9B's fix, automated); *p* lets the forecast lean on its own recent values (autoregression — 9B's memory, harnessed); *q* lets it lean on recent forecast errors (self-correction). **SARIMA** adds the same three dials at the seasonal lag (12). The full theory is a course; the *use* is four lines — and its report card, like everyone else's, is the leaderboard.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# A sane default for monthly seasonal business data: SARIMA(1,1,1)x(1,1,1,12)
sar = SARIMAX(train, order=(1,1,1), seasonal_order=(1,1,1,12)).fit(disp=False)
f_sar = sar.forecast(len(test))
print(eval_forecast(f_sar, test, "SARIMA").round(1).to_string())

In [ ]:
# THE LEADERBOARD - the only chart that matters
board = pd.DataFrame([eval_forecast(f_naive, test, "naive"),
                      eval_forecast(f_snaive, test, "seasonal naive  <- floor"),
                      eval_forecast(f_hw, test, "Holt-Winters"),
                      eval_forecast(f_sar, test, "SARIMA")]).round(1)
print(board.sort_values("RMSE").to_string())

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(sales.iloc[-30:], color="#94A3B8", lw=1, label="actual")
ax.plot(f_snaive, "s--", ms=3, color="#B45309", lw=1, label="seasonal naive")
ax.plot(f_hw, "o-", ms=3, color="#2563EB", lw=1.4, label="Holt-Winters")
ax.plot(f_sar, "^-", ms=3, color="#7C3AED", lw=1.2, label="SARIMA")
ax.axvline(train.index[-1], color="black", ls=":", lw=1)
ax.set_title("Test year: the models vs the truth", loc="left", fontweight="bold")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

Read it like Module 6 taught: the smart models **beat the seasonal-naive floor** — this series has learnable structure (trend + stable season), so unlike prices, sophistication pays here. That contrast IS the lab's thesis: *forecastability lives where structure is allowed to survive.*

---
## 4. Intervals that widen — the honest cone

In [ ]:
pred = sar.get_forecast(len(test))
mean, ci = pred.predicted_mean, pred.conf_int(alpha=0.05)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sales.iloc[-36:], color="#94A3B8", lw=1)
ax.plot(mean, color="#7C3AED", lw=1.6, label="SARIMA forecast")
ax.fill_between(mean.index, ci.iloc[:,0], ci.iloc[:,1], color="#7C3AED", alpha=0.15, label="95% interval")
ax.plot(test, "k.", ms=5, label="actual")
ax.axvline(train.index[-1], color="black", ls=":", lw=1)
ax.set_title("The cone of honesty: uncertainty GROWS with horizon", loc="left", fontweight="bold")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

w = (ci.iloc[:,1]-ci.iloc[:,0])
print(f"Interval width: 1 month ahead Rs {w.iloc[0]:,.0f} cr -> 12 months ahead Rs {w.iloc[-1]:,.0f} cr ({w.iloc[-1]/w.iloc[0]:.1f}x)")
print("Errors COMPOUND with horizon. Any 12-month forecast quoted with 1-month confidence is lying.")

**The widening cone is the forecast-slider widget's static twin** (on the course site, you drag the horizon and watch the cone breathe). Internalise the shape: near forecasts are claims; far forecasts are scenarios. Budget season's fight over "the number for next March" should be a fight over a cone.

### ✏️ Exercises
1. **Walk-forward, the lab way:** re-fit Holt-Winters each month over the last 18 months (train on everything before, forecast one step). Does its walk-forward MAE stay ahead of seasonal naive's, or was the single split flattering it?
2. **Break the models:** inject 9A-Exercise-3's level shift (+₹120 cr from month 40) into the training data and re-run the leaderboard. Which model recovers fastest from a break — and what does that suggest about smoothing's alpha as a "forgetting rate"?
3. **Forecast the un-forecastable:** point the identical SARIMA pipeline at NIFTY's monthly closes. Compare it to plain naive. Write the two-sentence conclusion this whole lab has been building toward.

---
## Lab 1 complete

You can now: decompose a series and audit the residual · measure memory against a noise band · test stationarity by eye and fix it by differencing · pick the honest floor · deploy smoothing and SARIMA inside Module 6's harness · and present a cone, never a line. **Badge: Time Bender ⏳**

*AI disclosure: ______*

In [ ]:
# workspace
